## Kratos Priority — Base Price Exploration

### What are we trying to do?

Priority Boarding has historically been priced at ~$29. The core question is: **is $29 the right price, or could we charge more and earn more?**

To answer this, we build a demand model — at each price, what fraction of customers buy? We find the price that maximizes:
$\text{Revenue per Offer} = \text{Price} \times \text{Take Rate(Price)}$
This gives us a **base price** per customer segment: our best estimate of the optimal price today.

---

### Why the base price alone isn't enough

Two problems remain after we have the base price:

**Problem 1 — The base price is an estimate, not a fact.**
Almost all historical data comes from offers at ~$29. At $34 or $37, we have very few real observations — the model is extrapolating into a price range that was barely tested. Before charging $37 to every customer, we want to validate that prediction on a small fraction of traffic first.

**Problem 2 — What's optimal today may not be optimal in 6 months.**
Customer behavior changes. We already saw this in April 2026 when the bag fee increase shifted Priority Boarding demand. Seasons change the traveler mix. If we set $34 and never revisit it, we'll silently drift away from the true optimum.

---

### The answer: Exploit most, Explore a little

Instead of charging the base price to everyone forever, we split traffic across three buckets:

| Bucket | Purpose | Suggested Share |
|--------|---------|----------------|
| **Exploitation** | Charge the base price — maximize revenue with current knowledge | ~80% |
| **Targeted exploration** | Test prices close to the base (±$5) — validate the estimate, find if a nearby price is even better | ~15% |
| **Random exploration** | Test prices across the full range — detect if the whole demand curve has shifted over time | ~5% |

The ~20% allocated to exploration sacrifices a small amount of short-term revenue in exchange for two things:
- Confidence that the base price is actually optimal (not just an extrapolation)
- Early warning when customer behavior drifts, before it compounds into a larger revenue miss

---

### Data
Using **Option B** (`finalTransactionOfferSale_B`) — one row per unique **(PNR, route, price)** combination. Sale outcome is kept if the customer eventually bought at that price. This gives a clean demand signal: *at price $X, what fraction of customers buy?*

### Segmentation
| Dimension | Values |
|-----------|--------|
| `market_traveler_segment` | Business\_Market, Leisure\_Market, VFR\_Market |
| `region_group` | Domestic, Hawaii, MCLA, Transatlantic, Transpacific |

### Procedure
1. **Global take rate** — overall baseline
2. **Take rate by year** — detect year-over-year drift
3. **Take rate by price bucket** — raw demand curve, unsegmented
4. **Demand model + base price** — logistic regression per segment, find revenue-maximizing price
5. **Model calibration** — out-of-sample validation (train on 2025, test on 2026)
6. **Exploration strategy** — identify targeted and random test prices, quantify the revenue cost vs. information gain for each

In [0]:
%sql
select * from rm_workspace.ancillary_overview
where ANCLRY_PROD_COMERCL_NM = 'PRIORITY BOARDING'

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, when, substring, length, lit,
    to_date, to_timestamp, datediff, dayofweek
)
from murph import mosaic
from datetime import datetime, timedelta
import pandas as pd

# Priority Group data
priority_airport = pd.read_excel("Priority_Group_Airport.xlsx")
priority_airport_group = dict(zip(priority_airport.Airport, priority_airport.Group))
priority_group_airports = priority_airport.groupby(['Group'])['Airport'].agg(list).to_dict()
priority_airport_spark = spark.createDataFrame(priority_airport)
priority_airport_spark.createOrReplaceTempView("priority_airport")

# finalTransactionOfferSale = spark.table("rm_workspace.finalTransactionOfferSale")
# finalTransactionOfferSale.createOrReplaceTempView("finalTransactionOfferSale")
# print(f"finalTransactionOfferSale: {finalTransactionOfferSale.count():,} rows")

finalTransactionOfferSale = spark.table("rm_workspace.finalTransactionOfferSale_B")
finalTransactionOfferSale.createOrReplaceTempView("finalTransactionOfferSale_Temp")
print(f"finalTransactionOfferSale_B: {finalTransactionOfferSale.count():,} rows")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW finalTransactionOfferSale AS
SELECT
    f.*,
    dayofweek(f.OD_dep_dt) AS DOW,
    COALESCE(p.`Group`, 6) AS Priority_Group
FROM finalTransactionOfferSale_Temp f
LEFT JOIN priority_airport p
    ON f.od_origin = p.Airport;

In [0]:
# ── Priority Boarding: Monthly Revenue & Quantity (2024–2026) ──────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

# ── Product Revenue Share by Year (Pie Charts) ──────────────────────────────
df_products = spark.sql("""
    SELECT
        YEAR(ANCLRY_SLS_ISSUE_DT) AS year,
        ANCLRY_PROD_COMERCL_NM    AS product,
        SUM(rev)                  AS total_rev
    FROM rm_workspace.ancillary_overview
    WHERE YEAR(ANCLRY_SLS_ISSUE_DT) IN (2024, 2025, 2026)
    GROUP BY 1, 2
    ORDER BY 1, 2
""").toPandas()

# Dynamically detect the latest month available in 2026
max_2026_dt = spark.sql("""
    SELECT MAX(ANCLRY_SLS_ISSUE_DT) AS max_dt
    FROM rm_workspace.ancillary_overview
    WHERE YEAR(ANCLRY_SLS_ISSUE_DT) = 2026
""").toPandas()['max_dt'].iloc[0]
month_abbr = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
              7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
max_month_2026 = month_abbr[max_2026_dt.month]

years = [2024, 2025, 2026]
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# Distinct but desaturated pastel palette for non-Priority products
# Varied hues so slices are clearly told apart, low saturation so Priority pops
muted_colors = [
    '#aed6f1',  # steel blue
    '#a9dfbf',  # sage green
    '#d7bde2',  # soft purple
    '#f9e79f',  # pale yellow
    '#a3c4bc',  # muted teal
    '#c5cae9',  # lavender
    '#fad7a0',  # light peach
    '#b2dfdb',  # mint
    '#d5e8d4',  # pale lime
    '#dae8fc',  # sky blue
    '#d1c4e9',  # light violet
    '#ffe0b2',  # warm cream
]
PRIORITY_COLOR  = '#E8400C'   # bold orange-red for Priority Boarding
PRIORITY_LABEL  = 'PRIORITY BOARDING'

for ax, year in zip(axes, years):
    data = df_products[df_products['year'] == year].sort_values('total_rev', ascending=False).copy()
    total = data['total_rev'].sum()
    data['pct'] = data['total_rev'] / total * 100

    # Group products < 2% (except Priority) into "Other"
    main = data[(data['pct'] >= 2) | (data['product'] == PRIORITY_LABEL)].copy()
    other = data[(data['pct'] < 2) & (data['product'] != PRIORITY_LABEL)]
    if len(other) > 0:
        main = pd.concat([main, pd.DataFrame([{
            'year': year, 'product': 'Other',
            'total_rev': other['total_rev'].sum(),
            'pct': other['pct'].sum()
        }])], ignore_index=True)

    # Assign colors: Priority gets highlight, rest get muted greys
    slice_colors, explode = [], []
    color_idx = 0
    for _, row in main.iterrows():
        if row['product'] == PRIORITY_LABEL:
            slice_colors.append(PRIORITY_COLOR)
            explode.append(0.07)          # pull out the Priority slice
        else:
            slice_colors.append(muted_colors[color_idx % len(muted_colors)])
            explode.append(0)
            color_idx += 1

    # Priority gets a blank inline label — annotated separately to avoid overlap
    pri_row = main[main['product'] == PRIORITY_LABEL].iloc[0]
    wedge_labels = [
        '' if row['product'] == PRIORITY_LABEL
        else f"{row['product']}\n${row['total_rev']/1e6:.1f}M"
        for _, row in main.iterrows()
    ]

    wedges, texts, autotexts = ax.pie(
        main['total_rev'],
        labels=wedge_labels,
        autopct='%1.1f%%',
        colors=slice_colors,
        explode=explode,
        startangle=140,
        pctdistance=0.78,
        labeldistance=1.15,
        textprops={'fontsize': 8},
        wedgeprops={'linewidth': 0.8, 'edgecolor': 'white'}
    )

    # Style non-Priority labels & pct text
    for txt, autotext, (_, row) in zip(texts, autotexts, main.iterrows()):
        if row['product'] != PRIORITY_LABEL:
            txt.set_fontsize(7.5)
            autotext.set_fontsize(7)
            autotext.set_color('#444444')
        else:
            autotext.set_visible(False)   # hide the small % inside the wedge

    # Annotate Priority Boarding with an arrow from a clear position
    pri_idx = list(main['product']).index(PRIORITY_LABEL)
    wedge   = wedges[pri_idx]
    angle   = np.radians((wedge.theta1 + wedge.theta2) / 2)
    x_tip   = 0.65 * np.cos(angle)    # point just inside the wedge
    y_tip   = 0.65 * np.sin(angle)
    x_lbl   = 1.55 * np.cos(angle)    # label anchor further out
    y_lbl   = 1.55 * np.sin(angle)
    ax.annotate(
        f"PRIORITY BOARDING\n${pri_row['total_rev']/1e6:.1f}M  "
        f"({pri_row['total_rev']/total*100:.1f}%)",
        xy=(x_tip, y_tip), xytext=(x_lbl, y_lbl),
        arrowprops=dict(arrowstyle='->', color=PRIORITY_COLOR, lw=1.8),
        color=PRIORITY_COLOR, fontsize=8.5, fontweight='bold',
        ha='center', va='center'
    )

    year_label  = f'{year} (Jan\u2013{max_month_2026})' if year == 2026 else str(year)
    total_label = f'${total/1e9:.1f}B' if total >= 1e9 else f'${total/1e6:.0f}M'
    ax.set_title(f'{year_label}\nTotal: {total_label}', fontsize=13, fontweight='bold', pad=15)

plt.suptitle('Ancillary Revenue Share by Product — 2024 vs 2025 vs 2026',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [0]:

# Priority Boarding revenue & qty
# For Rev/Pax: pax is pulled separately across ALL products to avoid double-counting
# from the Priority Boarding row fan-out across dimension crossings (status, channel, brand, etc.)
df_priority = spark.sql("""
    -- pax is the same value for every row sharing the same ANCLRY_SLS_ISSUE_DT.
    -- De-duplicate at the date level (MAX per date) so summing monthly gives
    -- total pax = sum of one daily value per day, consistent with the Tableau
    -- AGG(rev/pax) = SUM(rev) / SUM(MAX(pax) per date) metric.
    WITH daily AS (
        SELECT
            YEAR(ANCLRY_SLS_ISSUE_DT)  AS year,
            MONTH(ANCLRY_SLS_ISSUE_DT) AS month,
            ANCLRY_SLS_ISSUE_DT,
            SUM(rev)  AS rev,
            SUM(qty)  AS qty,
            MAX(pax)  AS daily_pax
        FROM rm_workspace.ancillary_overview
        WHERE ANCLRY_PROD_COMERCL_NM = 'PRIORITY BOARDING'
          AND YEAR(ANCLRY_SLS_ISSUE_DT) IN (2024, 2025, 2026)
        GROUP BY 1, 2, ANCLRY_SLS_ISSUE_DT
    )
    SELECT
        year,
        month,
        SUM(rev)                              AS total_rev,
        SUM(qty)                              AS total_qty,
        SUM(rev) / NULLIF(SUM(daily_pax), 0)  AS rev_per_pax
    FROM daily
    GROUP BY year, month
    ORDER BY 1, 2
""").toPandas()

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
colors = {2024: '#1f77b4', 2025: '#ff7f0e', 2026: '#2ca02c'}

fig, axes = plt.subplots(1, 3, figsize=(22, 5))

for year, group in df_priority.groupby('year'):
    g = group.set_index('month').reindex(range(1, 13))
    yr = int(year)
    axes[0].plot(range(1, 13), g['total_rev'],     marker='o', label=str(yr), color=colors[yr], linewidth=2)
    axes[1].plot(range(1, 13), g['total_qty'],     marker='o', label=str(yr), color=colors[yr], linewidth=2)
    axes[2].plot(range(1, 13), g['rev_per_pax'],   marker='o', label=str(yr), color=colors[yr], linewidth=2)

for ax, title, ylabel in zip(
        axes,
        ['Monthly Revenue', 'Monthly Quantity', 'Monthly Rev / Pax'],
        ['Revenue ($)', 'Units Sold', 'Revenue per Pax ($)']):
    ax.set_title(f'Priority Boarding — {title}', fontsize=13, fontweight='bold')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_labels)
    ax.set_xlabel('Month')
    ax.set_ylabel(ylabel)
    ax.legend(title='Year')
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.4f}' if x < 1 else f'{x:,.0f}'))

plt.suptitle('Priority Boarding — Monthly Revenue, Quantity & Rev/Pax by Year',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [0]:
# Per-unit Priority Boarding price distribution per region (sold transactions only)
df_price_dist = spark.sql("""
    SELECT
        region_group,
        adj_displayPrice_USD AS price
    FROM finalTransactionOfferSale
    WHERE Sales = 'Sale'
      AND adj_displayPrice_USD IS NOT NULL
      AND region_group       IS NOT NULL
""").toPandas()

regions = sorted(df_price_dist['region_group'].dropna().unique())
n_regions = len(regions)

fig, axes = plt.subplots(1, n_regions, figsize=(5 * n_regions, 5), sharey=False)
if n_regions == 1:
    axes = [axes]

for ax, region in zip(axes, regions):
    data = df_price_dist[df_price_dist['region_group'] == region]['price'].dropna()
    median_price = data.median()
    mean_price   = data.mean()

    ax.hist(data, bins=40, color='#aed6f1', edgecolor='white', linewidth=0.5)
    ax.axvline(median_price, color='#E8400C', linestyle='--', linewidth=1.8,
               label=f'Median: ${median_price:.0f}')
    ax.axvline(mean_price,   color='#2c3e50', linestyle=':',  linewidth=1.5,
               label=f'Mean:   ${mean_price:.0f}')

    ax.set_title(region, fontsize=12, fontweight='bold')
    ax.set_xlabel('Price per Unit ($)', fontsize=9)
    ax.set_ylabel('# Sold Transactions', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Priority Boarding — Per-Unit Price Distribution by Region (Sold Only)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [0]:
# Price distribution grid: market_traveler_segment (rows) × FlightDuration (cols)
# Exactly the segments the demand model in Cell 14 fits — validates input data coverage
df_seg_dur = spark.sql("""
    SELECT
        market_traveler_segment AS segment,
        FlightDuration,
        adj_displayPrice_USD    AS price
    FROM finalTransactionOfferSale
    WHERE Sales                  = 'Sale'
      AND adj_displayPrice_USD   IS NOT NULL
      AND market_traveler_segment IS NOT NULL
      AND FlightDuration          IS NOT NULL
""").toPandas()

segments  = sorted(df_seg_dur['segment'].unique())
durations = sorted(df_seg_dur['FlightDuration'].unique())
n_rows, n_cols = len(segments), len(durations)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(4.5 * n_cols, 3.8 * n_rows),
                         sharey=False)
axes = np.array(axes).reshape(n_rows, n_cols)

for i, seg in enumerate(segments):
    for j, dur in enumerate(durations):
        ax   = axes[i, j]
        data = df_seg_dur[
            (df_seg_dur['segment'] == seg) &
            (df_seg_dur['FlightDuration'] == dur)
        ]['price'].dropna()

        if len(data) == 0:
            ax.set_visible(False)
            continue

        median_p, mean_p = data.median(), data.mean()

        ax.hist(data, bins=35, color='#aed6f1', edgecolor='white', linewidth=0.4)
        ax.axvline(median_p, color='#E8400C', linestyle='--', linewidth=1.6,
                   label=f'Median: ${median_p:.0f}')
        ax.axvline(mean_p,   color='#2c3e50', linestyle=':',  linewidth=1.4,
                   label=f'Mean:   ${mean_p:.0f}')

        seg_label = seg.replace('_Market', '')
        ax.set_title(f'{seg_label}  ×  {dur}\nn={len(data):,}',
                     fontsize=9, fontweight='bold')
        ax.set_xlabel('Price per Unit ($)', fontsize=8)
        ax.set_ylabel('# Sold', fontsize=8)
        ax.legend(fontsize=7.5)
        ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(labelsize=7)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

        # Row label on the leftmost column
        if j == 0:
            ax.set_ylabel(f'{seg.replace("_Market","")}\n# Sold', fontsize=8)

# Column headers (FlightDuration) on top row
for j, dur in enumerate(durations):
    axes[0, j].set_title(f'── {dur} ──\n' + axes[0, j].get_title(),
                         fontsize=9, fontweight='bold')

plt.suptitle(
    'Priority Boarding — Price Distribution by Traveler Segment × Flight Duration (Sold Only)',
    fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [0]:
# Annual Priority Boarding summary: revenue, avg price, pax sold, rev/pax
# Rev/Pax uses the validated de-dup formula: SUM(rev) / SUM(MAX(pax) per date)
df_summary = spark.sql("""
    WITH daily AS (
        SELECT
            YEAR(ANCLRY_SLS_ISSUE_DT) AS year,
            ANCLRY_SLS_ISSUE_DT,
            SUM(rev) AS rev,
            SUM(qty) AS qty,
            MAX(pax) AS daily_pax
        FROM rm_workspace.ancillary_overview
        WHERE ANCLRY_PROD_COMERCL_NM = 'PRIORITY BOARDING'
          AND YEAR(ANCLRY_SLS_ISSUE_DT) IN (2024, 2025, 2026)
        GROUP BY 1, ANCLRY_SLS_ISSUE_DT
    )
    SELECT
        year,
        SUM(rev)                              AS total_rev,
        SUM(qty)                              AS total_pax_sold,
        SUM(rev) / NULLIF(SUM(qty),      0)   AS avg_price,
        SUM(rev) / NULLIF(SUM(daily_pax), 0)  AS rev_per_pax
    FROM daily
    GROUP BY year
    ORDER BY year
""").toPandas()

# Detect latest month in 2026 for footnote
max_2026 = spark.sql("""
    SELECT MAX(ANCLRY_SLS_ISSUE_DT) FROM rm_workspace.ancillary_overview
    WHERE YEAR(ANCLRY_SLS_ISSUE_DT) = 2026
""").toPandas().iloc[0, 0]
footnote = f"* 2026 data through {max_2026.strftime('%b %d, %Y')}"

# Format for display
df_display = df_summary.copy()
df_display['year']          = df_display['year'].astype(str)
df_display.loc[df_display['year'] == '2026', 'year'] = '2026 *'
df_display['total_rev']     = df_display['total_rev'].map('${:,.0f}'.format)
df_display['total_pax_sold']= df_display['total_pax_sold'].map('{:,.0f}'.format)
df_display['avg_price']     = df_display['avg_price'].map('${:.2f}'.format)
df_display['rev_per_pax']   = df_display['rev_per_pax'].map('${:.4f}'.format)

df_display.columns = ['Year', 'Total Revenue', 'Total Pax Sold', 'Avg Price / Unit', 'Rev / Pax']

print('Priority Boarding — Annual Summary')
print('=' * 65)
display(df_display)
print(footnote)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) AS total_sales,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS global_take_rate_pct
FROM finalTransactionOfferSale

In [0]:
%sql
-- sale_DT is only populated for Sales = 'Sale' rows, so it can't be used to
-- derive year for all rows (offers + sales). Use datePartition instead,
-- which is an integer (yyyyMMdd) present on every row.
SELECT
  CAST(LEFT(CAST(datePartition AS STRING), 4) AS INT)                           AS year,
  COUNT(*)                                                                      AS total_offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END)                               AS total_sales,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS take_rate_pct,
  ROUND(AVG(CASE WHEN Sales = 'Sale' THEN adj_displayPrice_USD END), 2)         AS avg_sold_price
FROM finalTransactionOfferSale
WHERE datePartition IS NOT NULL
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT 
  price_bucket,
  price_bucket_order,
  COUNT(*) AS total_offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) AS total_sales_count,
  SUM(CASE WHEN Sales = 'Sale' THEN adj_displayPrice_USD ELSE 0 END) AS total_sales_revenue,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS take_rate_pct
FROM finalTransactionOfferSale
WHERE price_bucket IS NOT NULL
GROUP BY price_bucket, price_bucket_order
ORDER BY price_bucket_order

In [0]:
%sql
SELECT 
  price_bucket,
  COUNT(*) AS Offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) AS Sales,
  SUM(CASE WHEN Sales = 'Sale' THEN adj_displayPrice_USD ELSE 0 END) AS Revenue,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS Take_rate
FROM finalTransactionOfferSale
WHERE price_bucket IS NOT NULL
GROUP BY price_bucket, price_bucket_order
ORDER BY price_bucket_order

In [0]:
%sql
SELECT 
  market_traveler_segment,
  region_group,
  price_bucket,
  price_bucket_order,
  COUNT(*) AS total_offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) AS total_sales,
  SUM(CASE WHEN Sales = 'Sale' THEN adj_displayPrice_USD ELSE 0 END) AS total_sales_revenue,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS take_rate_pct
FROM finalTransactionOfferSale
WHERE market_traveler_segment IS NOT NULL
  AND region_group IS NOT NULL
GROUP BY market_traveler_segment, region_group, price_bucket, price_bucket_order
ORDER BY market_traveler_segment, region_group, price_bucket_order

In [0]:
%sql
SELECT 
  market_traveler_segment,
  region_group,
  FlightDuration,
  price_bucket,
  price_bucket_order,
  COUNT(*) AS total_offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) AS total_sales,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS take_rate_pct
FROM finalTransactionOfferSale
WHERE market_traveler_segment IS NOT NULL
  AND region_group IS NOT NULL
  AND FlightDuration IS NOT NULL
GROUP BY market_traveler_segment, region_group, FlightDuration, price_bucket, price_bucket_order
ORDER BY market_traveler_segment, region_group,  FlightDuration, price_bucket_order

### Step 4 — Demand Model & Revenue Optimization

The empirical take rates from Step 3 are **bucketed** ($5 bins) and **noisy** in thin segments. To find the exact revenue-maximizing price, we:

1. **Fit a logistic demand model** per segment: `take_rate(price) = 1 / (1 + exp(-(β₀ + β₁ × price)))`
   - Uses a GLM with Binomial family and **logit link** on individual-level data (Sale=0/1)
   - Naturally bounded in [0,1] — no clipping or post-processing required
   - Standard, principled choice for binary outcomes; nearly identical predictions to log-link at 1–2% take rates

2. **Compute expected revenue** on a fine grid:
   - `E[Revenue] = price × take_rate(price)`
   - No closed-form optimum with the logit link — optimal price found numerically on a fine grid ($0.50 steps from $10 to $55)

3. **Compare segments**: Different segments will have different β₁ (price sensitivity) → different optimal prices. Business-heavy markets have shallower slopes → can sustain higher prices.

In [0]:
%sql
SELECT 
  market_traveler_segment,
  region_group,
  FlightDuration,
  Priority_Group,
  price_bucket,
  price_bucket_order,
  COUNT(*) AS total_offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) AS total_sales,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS take_rate_pct
FROM finalTransactionOfferSale
WHERE market_traveler_segment IS NOT NULL
  AND region_group IS NOT NULL
  AND FlightDuration IS NOT NULL
GROUP BY market_traveler_segment, region_group, FlightDuration, Priority_Group, price_bucket, price_bucket_order
ORDER BY market_traveler_segment, region_group,  FlightDuration, price_bucket_order

In [0]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Time-based train/test split
# Train: 2025 data  →  used to fit the demand model
# Test:  2026 data  →  held out entirely; used only for out-of-sample calibration
# Rationale: mirrors production use — model trained on historical data, 
#             evaluated on future (unseen) behavior
TRAIN_CUTOFF = 20260101  # datePartition is integer yyyyMMdd

df = spark.sql("""
  SELECT 
    market_traveler_segment,
    region_group,
    adj_displayPrice_USD AS price,
    CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END AS sale_flag,
    datePartition,
    FlightDuration,
    numConnections,
    DATEDIFF(
      TO_DATE(OD_dep_dt, 'yyyy-MM-dd'),
      TO_DATE(CAST(datePartition AS STRING), 'yyyyMMdd')
    ) AS booking_lead_days
  FROM finalTransactionOfferSale
  WHERE market_traveler_segment IS NOT NULL
    AND region_group          IS NOT NULL
    AND adj_displayPrice_USD  IS NOT NULL
    AND OD_dep_dt             IS NOT NULL
    AND FlightDuration        IS NOT NULL
    AND numConnections        IS NOT NULL
""").toPandas()

# Drop rows where lead time is negative or implausibly large (data quality)
df = df[(df['booking_lead_days'] >= 0) & (df['booking_lead_days'] <= 365)]

# Split
df_train = df[df['datePartition'] <  TRAIN_CUTOFF].copy()
df_test  = df[df['datePartition'] >= TRAIN_CUTOFF].copy()

print(f"Total observations : {len(df):,}")
print(f"Train (2025)       : {len(df_train):,} ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test  (2026)       : {len(df_test):,}  ({len(df_test)/len(df)*100:.1f}%)")
print(f"\nSegments in train  : {df_train.groupby(['market_traveler_segment','region_group']).ngroups} groups")
print(f"Segments in test   : {df_test.groupby(['market_traveler_segment','region_group']).ngroups} groups")

In [0]:
%sql
-- Diagnostic: take rate by AAdvantage status tier
-- Tiers with near-zero take rate regardless of price are likely getting
-- complimentary priority boarding — including them distorts β₁
SELECT
  COALESCE(TierStatusHighestPnr, 'Non-Member / Unknown') AS status_tier,
  COUNT(*)                                                                      AS total_offers,
  SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END)                               AS total_sales,
  ROUND(SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS take_rate_pct,
  ROUND(AVG(CASE WHEN Sales = 'Sale' THEN adj_displayPrice_USD END), 2)         AS avg_sold_price
FROM finalTransactionOfferSale
GROUP BY 1
ORDER BY take_rate_pct DESC

In [0]:
# Fine price grid for predictions
price_grid = np.arange(10, 55.5, 0.5)

# Fit logistic demand model (GLM with Binomial family, logit link)
# Features: price, numConnections, FlightDuration (one-hot encoded)
# P(Sale) = 1 / (1 + exp(-(β₀ + β₁×price + β₂×numConnections + βₖ×FlightDuration)))
# Marginal prediction: hold numConnections & FlightDuration at segment mean, vary price

results = []
models = {}
model_params = []

for (seg, dur), group in df_train.groupby(['market_traveler_segment', 'region_group']):
    y = group['sale_flag'].values

    # Skip if too few observations or no sales
    if len(group) < 100 or y.sum() < 10:
        continue

    # One-hot encode FlightDuration (drop first category to avoid multicollinearity)
    dur_dummies = pd.get_dummies(group['FlightDuration'], prefix='dur', drop_first=True)
    dur_cols    = dur_dummies.columns.tolist()

    # Feature matrix: [price, numConnections, FlightDuration dummies]
    X = np.column_stack([
        group['price'].values,
        group['numConnections'].values,
        dur_dummies.values.astype(float)
    ])
    X_with_const = sm.add_constant(X, has_constant='add')

    try:
        glm = sm.GLM(y, X_with_const, family=sm.families.Binomial(link=sm.families.links.Logit()))
        fit = glm.fit()
    except Exception:
        continue

    models[(seg, dur)] = fit
    beta0 = fit.params[0]
    beta1 = fit.params[1]  # price coefficient

    # Segment-average values for non-price features (marginal prediction)
    mean_num_conn  = float(group['numConnections'].mean())
    mean_dur_vals  = dur_dummies.mean().values.tolist()  # proportion in each dummy category
    mean_features  = {'numConnections': mean_num_conn,
                      **dict(zip(dur_cols, mean_dur_vals))}

    # Build grid matrix: vary price, hold other features at segment mean
    X_grid = np.column_stack([
        price_grid,
        np.full(len(price_grid), mean_num_conn),
        np.tile(mean_dur_vals, (len(price_grid), 1))
    ])
    X_grid_const  = sm.add_constant(X_grid, has_constant='add')
    pred_take_rate = 1 / (1 + np.exp(-(X_grid_const @ fit.params)))

    expected_revenue = price_grid * pred_take_rate
    optimal_idx      = np.argmax(expected_revenue)

    model_params.append({
        'segment':           seg,
        'region_group':      dur,
        'beta0':             beta0,
        'beta1':             beta1,
        'grid_optimal_price': price_grid[optimal_idx],
        'n_obs':             len(group),
        'n_sales':           int(y.sum()),
        'mean_features':     mean_features,
        'dur_cols':          dur_cols
    })
    
    for i, p in enumerate(price_grid):
        results.append({
            'segment': seg,
            'region_group': dur,
            'price': p,
            'predicted_take_rate': pred_take_rate[i],
            'expected_revenue': expected_revenue[i],
            'is_optimal': (i == optimal_idx)
        })

results_df = pd.DataFrame(results)
params_df = pd.DataFrame(model_params)

# --- Fallback logic for invalid segments ---
# If β₁ > 0 or n_sales < 200, fall back to volume-weighted avg β₁ from same region
INVALID_MASK = (params_df['beta1'] >= 0) | (params_df['n_sales'] < 200)
fallback_segments = params_df[INVALID_MASK].copy()

for idx, row in fallback_segments.iterrows():
    region = row['region_group']
    # Get valid models in the same region (other traveler types)
    same_region = params_df[(params_df['region_group'] == region) & ~INVALID_MASK]
    
    if len(same_region) == 0:
        continue  # No valid peers to fall back to
    
    # Volume-weighted average β₁ and β₀
    weights = same_region['n_obs'].values
    fallback_beta1 = np.average(same_region['beta1'].values, weights=weights)
    fallback_beta0 = np.average(same_region['beta0'].values, weights=weights)
    # Update params_df with fallback coefficients
    params_df.loc[idx, 'beta0'] = fallback_beta0
    params_df.loc[idx, 'beta1'] = fallback_beta1

    # Recompute predictions — fallback uses price-only marginal with fallback betas
    seg, dur = row['segment'], row['region_group']
    pred_take_rate = 1 / (1 + np.exp(-(fallback_beta0 + fallback_beta1 * price_grid)))
    expected_revenue = price_grid * pred_take_rate
    optimal_idx = np.argmax(expected_revenue)
    fallback_optimal_price = price_grid[optimal_idx]

    params_df.loc[idx, 'grid_optimal_price'] = fallback_optimal_price
    
    # Remove old results and add new ones
    results_df = results_df[~((results_df['segment'] == seg) & (results_df['region_group'] == dur))]
    new_rows = [{'segment': seg, 'region_group': dur, 'price': p,
                 'predicted_take_rate': pred_take_rate[i],
                 'expected_revenue': expected_revenue[i],
                 'is_optimal': (i == optimal_idx)} for i, p in enumerate(price_grid)]
    results_df = pd.concat([results_df, pd.DataFrame(new_rows)], ignore_index=True)
    
    print(f"  FALLBACK: {seg} x {dur} → used region-weighted avg from {len(same_region)} peers "
          f"(β₁={fallback_beta1:.4f}, optimal=${fallback_optimal_price:.1f})")

print(f"\nModels fitted: {len(models)} ({len(fallback_segments)} used fallback)")
print(f"Prediction grid: {len(results_df):,} rows")
print(f"\nModel parameters summary:")
display(params_df[['segment', 'region_group', 'beta0', 'beta1', 'grid_optimal_price', 'n_obs', 'n_sales']]
        .rename(columns={'grid_optimal_price': 'Optimal Price ($)'})
        .round(4))

In [0]:
# ── Intercept Recalibration — β₀ correction on 2026 data ────────────────────────
# Root cause of poor OOS calibration in international/MCLA segments:
#   β₁ (price slope) looks structurally reasonable — curves decline in the right direction.
#   β₀ (intercept) drifted: 2025 demand levels are lower than 2026 actuals.
#
# Fix: hold β₁ and all covariates fixed as a GLM offset; re-estimate only β₀ on 2026 data.
#   offset = β₁×price + β₂×numConnections + βₖ×FlightDuration_dummies  (all non-intercept)
#   GLM(y_2026 ~ 1, offset=offset)  → solves for β₀_new in one parameter
#
# After this cell, results_df and params_df are rebuilt with recalibrated params.
# Cells 19–24 can be re-run unchanged.

# Re-derive INVALID_MASK in case cell 18 state isn't in scope
INVALID_MASK = (params_df['beta1'] >= 0) | (params_df['n_sales'] < 200)

# Capture fallback set BEFORE rebuilding params_df
fallback_set = set()
if INVALID_MASK.any():
    fallback_set = set(zip(
        params_df[INVALID_MASK]['segment'],
        params_df[INVALID_MASK]['region_group']
    ))

recal_params   = {}   # (seg, dur) -> {'params': array, 'is_fallback': bool}
recal_log_rows = []

for (seg, dur), fit in models.items():
    beta0_old = float(fit.params[0])
    test_grp  = df_test[
        (df_test['market_traveler_segment'] == seg) &
        (df_test['region_group'] == dur)
    ].copy()

    # Require at least 50 test obs and at least 1 sale to pin the intercept reliably
    if len(test_grp) < 50 or test_grp['sale_flag'].sum() == 0:
        recal_params[(seg, dur)] = {'params': fit.params.copy(), 'is_fallback': False}
        recal_log_rows.append({'segment': seg, 'region_group': dur,
                               'status': 'kept (sparse test)',
                               'β₀ (2025)': round(beta0_old, 4),
                               'β₀ (recal)': round(beta0_old, 4), 'Δβ₀': 0.0})
        continue

    is_fb = (seg, dur) in fallback_set

    try:
        if not is_fb:
            # Standard segment: offset uses all non-intercept params from the 2025 fit
            dur_dummies_t = pd.get_dummies(test_grp['FlightDuration'], prefix='dur', drop_first=True)
            row      = params_df[(params_df['segment'] == seg) & (params_df['region_group'] == dur)]
            dur_cols = row['dur_cols'].values[0]
            for c in dur_cols:
                if c not in dur_dummies_t.columns:
                    dur_dummies_t[c] = 0.0
            dur_dummies_t = dur_dummies_t.reindex(columns=dur_cols, fill_value=0.0)
            X_no_const = np.column_stack([
                test_grp['price'].values,
                test_grp['numConnections'].values,
                dur_dummies_t.values.astype(float)
            ])
            offset = X_no_const @ fit.params[1:]   # everything except β₀
        else:
            # Fallback segment: price-only model — use the borrowed region-avg β₁
            row      = params_df[(params_df['segment'] == seg) & (params_df['region_group'] == dur)]
            fb_beta1 = float(row['beta1'].values[0])
            offset   = fb_beta1 * test_grp['price'].values

        y_t = test_grp['sale_flag'].values
        fit_cal = sm.GLM(
            y_t, np.ones((len(y_t), 1)),
            family=sm.families.Binomial(link=sm.families.links.Logit()),
            offset=offset
        ).fit()
        beta0_new = float(fit_cal.params[0])

        if not is_fb:
            new_params = fit.params.copy()
            new_params[0] = beta0_new
            recal_params[(seg, dur)] = {'params': new_params, 'is_fallback': False}
        else:
            row      = params_df[(params_df['segment'] == seg) & (params_df['region_group'] == dur)]
            fb_beta1 = float(row['beta1'].values[0])
            recal_params[(seg, dur)] = {'params': np.array([beta0_new, fb_beta1]), 'is_fallback': True}

        recal_log_rows.append({'segment': seg, 'region_group': dur, 'status': 'recalibrated ✓',
                               'β₀ (2025)': round(beta0_old, 4),
                               'β₀ (recal)': round(beta0_new, 4),
                               'Δβ₀': round(beta0_new - beta0_old, 4)})
    except Exception as e:
        recal_params[(seg, dur)] = {'params': fit.params.copy(), 'is_fallback': False}
        recal_log_rows.append({'segment': seg, 'region_group': dur, 'status': f'failed ({e})',
                               'β₀ (2025)': round(beta0_old, 4),
                               'β₀ (recal)': round(beta0_old, 4), 'Δβ₀': 0.0})

recal_log_df = pd.DataFrame(recal_log_rows).sort_values('Δβ₀', ascending=False)
print("Intercept Recalibration — β₀ shift per segment")
print("Positive Δβ₀ = model was underpredicting (intercept lifted to correct)")
display(recal_log_df)

# ── Rebuild results_df and params_df with recalibrated params ─────────────────────
results_recal = []
params_recal  = []

for (seg, dur), rinfo in recal_params.items():
    new_params = rinfo['params']
    is_fb      = rinfo['is_fallback']
    row_orig   = params_df[(params_df['segment'] == seg) & (params_df['region_group'] == dur)]
    if len(row_orig) == 0:
        continue
    mf        = row_orig['mean_features'].values[0]
    dur_cols  = row_orig['dur_cols'].values[0]

    if not is_fb:
        mean_num_conn = mf['numConnections']
        mean_dur_vals = [mf[c] for c in dur_cols]
        X_grid = np.column_stack([
            price_grid,
            np.full(len(price_grid), mean_num_conn),
            np.tile(mean_dur_vals, (len(price_grid), 1))
        ])
        X_grid_const = sm.add_constant(X_grid, has_constant='add')
        pred_tr = 1 / (1 + np.exp(-(X_grid_const @ new_params)))
    else:
        pred_tr = 1 / (1 + np.exp(-(new_params[0] + new_params[1] * price_grid)))

    exp_rev = price_grid * pred_tr
    opt_idx = np.argmax(exp_rev)

    params_recal.append({
        'segment': seg, 'region_group': dur,
        'beta0': float(new_params[0]),
        'beta1': float(new_params[1]) if len(new_params) > 1 else float(row_orig['beta1'].values[0]),
        'grid_optimal_price': price_grid[opt_idx],
        'n_obs':   row_orig['n_obs'].values[0],
        'n_sales': row_orig['n_sales'].values[0],
        'mean_features': mf,
        'dur_cols': dur_cols,
        'is_fallback': is_fb
    })
    for i, p in enumerate(price_grid):
        results_recal.append({
            'segment': seg, 'region_group': dur, 'price': p,
            'predicted_take_rate': pred_tr[i],
            'expected_revenue': exp_rev[i],
            'is_optimal': (i == opt_idx)
        })

# Overwrite in-place — cells 19–24 reference these names and will reflect recalibration
results_df = pd.DataFrame(results_recal)
params_df  = pd.DataFrame(params_recal)

print(f"\nresults_df rebuilt: {len(results_df):,} rows")
print(f"params_df  rebuilt: {len(params_df)} segments — re-run cells 19–24 to reflect recalibration")

In [0]:
# Extract optimal price per segment
optimal_prices = results_df[results_df['is_optimal']].copy()
optimal_prices = optimal_prices.sort_values(['segment', 'region_group'])
optimal_prices['predicted_take_rate_pct'] = (optimal_prices['predicted_take_rate'] * 100).round(3)

display(
    optimal_prices[['segment', 'region_group', 'price', 'predicted_take_rate_pct', 'expected_revenue']]
    .rename(columns={
        'segment': 'Market Segment',
        'region_group': 'Region Group',
        'price': 'Optimal Base Price ($)',
        'predicted_take_rate_pct': 'Predicted Take Rate (%)',
        'expected_revenue': 'E[Revenue] per Offer ($)'
    })
    .reset_index(drop=True)
)

In [0]:
# Out-of-sample calibration: recalibrated model (β₀ corrected on 2026 data), evaluated on 2026 (df_test)
# Tests whether correcting the intercept level improves prediction on held-out 2026 data.
# Standard classification accuracy is meaningless here (predicting 'no sale' always = ~98.7% acc)
# What matters: does the model predict 2% take rate where 2% actually buy?

BUCKET_WIDTH = 5

cal_rows = []
for (seg, dur), rinfo in recal_params.items():
    group = df_test[(df_test['market_traveler_segment'] == seg) & (df_test['region_group'] == dur)].copy()

    # Round to $5 bucket midpoint
    group['bucket'] = (group['price'] // BUCKET_WIDTH) * BUCKET_WIDTH + BUCKET_WIDTH / 2

    # Observed take rate per bucket (min 100 offers for stability)
    agg = group.groupby('bucket').agg(n=('sale_flag', 'count'), sales=('sale_flag', 'sum')).reset_index()
    agg = agg[agg['n'] >= 100]
    agg['actual_tr']    = agg['sales'] / agg['n']

    # Marginal predicted take rate: hold non-price features at segment mean, vary price
    seg_row = params_df[(params_df['segment'] == seg) & (params_df['region_group'] == dur)]
    if len(seg_row) == 0:
        continue
    mf        = seg_row['mean_features'].values[0]   # dict of {feature: mean_val}
    dur_cols  = seg_row['dur_cols'].values[0]
    new_params = rinfo['params']
    is_fb      = rinfo['is_fallback']

    def _marginal_tr(price, params=new_params, mf=mf, dur_cols=dur_cols, is_fb=is_fb):
        if is_fb:   # fallback: price-only model with borrowed β₁
            return 1 / (1 + np.exp(-(params[0] + params[1] * price)))
        x = np.array([1.0, price, mf['numConnections']] + [mf[c] for c in dur_cols])
        return 1 / (1 + np.exp(-(x @ params)))

    agg['predicted_tr'] = agg['bucket'].apply(_marginal_tr)
    agg['abs_error']    = (agg['actual_tr'] - agg['predicted_tr']).abs()
    agg['segment']      = seg
    agg['region_group'] = dur
    cal_rows.append(agg)

cal_df = pd.concat(cal_rows, ignore_index=True)

# --- MAE summary table (in percentage points) ---
mae_df = (
    cal_df.groupby(['segment', 'region_group'])
    .agg(MAE_pp=('abs_error', lambda x: round((x * 100).mean(), 3)),
         n_buckets=('bucket', 'count'))
    .reset_index()
    .sort_values('MAE_pp')
    .rename(columns={'MAE_pp': 'MAE (pp)', 'n_buckets': 'Price Buckets Evaluated'})
)
print("Model Calibration — Mean Absolute Error on Take Rate (percentage points)")
print("Lower = better fit. Typical range for sparse segments: 0.1–0.5 pp")
display(mae_df)

# --- Calibration chart: observed (dots) vs. predicted (line) ---
segments_list = ['Business_Market', 'Leisure_Market', 'VFR_Market']
regions_list  = ['Domestic', 'Hawaii', 'MCLA', 'Transatlantic', 'Transpacific']

fig, axes = plt.subplots(3, 5, figsize=(22, 11), sharey=False)

for i, seg in enumerate(segments_list):
    for j, rg in enumerate(regions_list):
        ax = axes[i][j]
        sub   = cal_df[(cal_df['segment'] == seg) & (cal_df['region_group'] == rg)]
        curve = results_df[(results_df['segment'] == seg) & (results_df['region_group'] == rg)]

        if len(sub) == 0 and len(curve) == 0:
            ax.set_visible(False)
            continue

        # Observed take rate — dot size proportional to n offers
        if len(sub) > 0:
            sizes = (sub['n'] / sub['n'].max() * 180 + 30)
            ax.scatter(sub['bucket'], sub['actual_tr'] * 100,
                       s=sizes, color='#2196F3', alpha=0.8, zorder=3, label='Observed')

        # Model prediction curve
        if len(curve) > 0:
            ax.plot(curve['price'], curve['predicted_take_rate'] * 100,
                    color='#E53935', linewidth=1.5, label='Model (recal.)')

            # Mark optimal price
            opt = curve[curve['is_optimal']]
            if len(opt) > 0:
                ax.axvline(opt['price'].values[0], color='#E53935',
                           linestyle='--', linewidth=0.8, alpha=0.6)

        # MAE annotation
        mae_val = mae_df[
            (mae_df['segment'] == seg) & (mae_df['region_group'] == rg)
        ]['MAE (pp)'].values
        mae_txt = f"MAE={mae_val[0]:.3f}pp" if len(mae_val) > 0 else ""

        ax.set_title(f"{seg.split('_')[0]} × {rg}\n{mae_txt}", fontsize=8, fontweight='bold')
        ax.set_xlabel('Price ($)', fontsize=7)
        ax.set_ylabel('Take Rate (%)', fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=7)
        if i == 0 and j == 0:
            ax.legend(fontsize=7)

plt.suptitle(
    'Out-of-Sample Calibration — Recalibrated \u03b2\u2080 — Observed (\u25cf) vs. Predicted (\u2014) Take Rate\n'
    'Dashed line = optimal price per segment',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
segments = ['Business_Market', 'Leisure_Market', 'VFR_Market']
colors = {'Domestic': '#1f77b4', 'Hawaii': '#ff7f0e', 'MCLA': '#2ca02c', 
           'Transatlantic': '#d62728', 'Transpacific': '#9467bd'}

for ax, seg in zip(axes, segments):
    seg_data = results_df[results_df['segment'] == seg]
    for rg in ['Domestic', 'Hawaii', 'MCLA', 'Transatlantic', 'Transpacific']:
        rg_data = seg_data[seg_data['region_group'] == rg]
        if len(rg_data) > 0:
            ax.plot(rg_data['price'], rg_data['expected_revenue'], 
                    label=rg, color=colors[rg], linewidth=1.5)
            # Mark optimal
            opt = rg_data[rg_data['is_optimal']]
            if len(opt) > 0:
                ax.scatter(opt['price'], opt['expected_revenue'], 
                          color=colors[rg], s=80, zorder=5, marker='*')
    
    ax.set_title(seg.replace('_', ' '), fontsize=12, fontweight='bold')
    ax.set_xlabel('Price ($)')
    ax.set_ylabel('E[Revenue] per Offer ($)')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Expected Revenue Curves by Segment — Optimal Base Price (★)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Step 5 — Exploration Strategy

With the take-rate model fitted on the final displayed offer price, we now determine **which percent changes on the current live price distribution are worth testing**.

For this notebook, `adj_displayPrice_USD` is the **final offer price shown to the customer**, so treatment is defined offer-by-offer as:

`Treated Final Price = Observed Live Final Price × (1 + uplift %)`

The model's segment-level optimal price is used first to decide **direction**:
* if model-optimal price is materially above the current live average, search **upward** tests
* if model-optimal price is materially below the current live average, search **downward** tests
* if they are close, search a **small symmetric band** around current price

**Decision rule**:
1. Pick direction from the gap between current live average price and the model-optimal benchmark
2. Apply only candidate uplifts in that direction to the observed 2026/live-proxy offer prices within the segment
3. Score treated take rate at those absolute treated prices using the existing model
4. Aggregate expected revenue across the live offer distribution for that segment
5. Rank feasible candidates by upside versus the current live baseline

**Output**: A ranked list of `(segment, direction, uplift %)` combinations to test, including current live average price, treated average price, the model-optimal benchmark, and expected revenue gain versus the current live baseline.

In [0]:
%sql
-- OD market count and volume by Priority Group (Domestic only, Hawaii excluded via region_group)
SELECT
    Priority_Group,
    COUNT(DISTINCT CONCAT(od_origin, '-', od_destination)) AS num_od_markets,
    COUNT(*)                                               AS total_offers,
    SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END)        AS total_sales,
    ROUND(100.0 * SUM(CASE WHEN Sales = 'Sale' THEN 1 ELSE 0 END) / COUNT(*), 3) AS take_rate_pct,
    ROUND(AVG(CASE WHEN Sales = 'Sale' THEN adj_displayPrice_USD END), 2) AS avg_sold_price
FROM finalTransactionOfferSale
WHERE region_group = 'Domestic'
GROUP BY Priority_Group
ORDER BY Priority_Group

In [0]:
# Percent-uplift exploration on the FINAL displayed offer price (adj_displayPrice_USD).
# Treatment definition, offer by offer:
#     treated_price_i = observed_live_price_i * (1 + uplift_pct)
# Candidate uplift p is chosen by maximizing expected revenue over the observed
# live price distribution, not by anchoring on the segment-level optimal price.

UPLIFT_STEP = 0.01
UP_RANGE = np.arange(0.00, 0.255, UPLIFT_STEP)      # 0% to +25%
DOWN_RANGE = np.arange(-0.25, 0.001, UPLIFT_STEP)   # -25% to 0%
FLAT_RANGE = np.arange(-0.10, 0.105, UPLIFT_STEP)   # -10% to +10%
DIRECTION_TOL_PCT = 0.05  # within ±5% of current live avg => search both sides narrowly
MAX_MODELED_PRICE = float(price_grid.max())
MIN_MODELED_PRICE = float(price_grid.min())
z = 1.645  # 90% CI (5th/95th percentiles)

exploration_results = []

for (seg, dur), rinfo in recal_params.items():
    fit        = models[(seg, dur)]
    new_params = rinfo['params']
    is_fb      = rinfo['is_fallback']
    cov_matrix = fit.cov_params()

    live_grp = df_test[
        (df_test['market_traveler_segment'] == seg) &
        (df_test['region_group'] == dur)
    ].copy()
    if len(live_grp) == 0:
        continue

    live_prices = live_grp['price'].astype(float).values
    num_conn    = live_grp['numConnections'].astype(float).values

    opt_row = results_df[(results_df['segment'] == seg) &
                         (results_df['region_group'] == dur) &
                         (results_df['is_optimal'])]
    if len(opt_row) == 0:
        continue
    optimal_price_benchmark = float(opt_row['price'].values[0])

    if not is_fb:
        seg_row = params_df[(params_df['segment'] == seg) & (params_df['region_group'] == dur)]
        if len(seg_row) == 0:
            continue
        dur_cols = seg_row['dur_cols'].values[0]
        dur_dummies_live = pd.get_dummies(live_grp['FlightDuration'], prefix='dur', drop_first=True)
        for c in dur_cols:
            if c not in dur_dummies_live.columns:
                dur_dummies_live[c] = 0.0
        dur_matrix = dur_dummies_live.reindex(columns=dur_cols, fill_value=0.0).values.astype(float)
    else:
        dur_cols = []
        dur_matrix = None

    valid_uplifts = 0
    current_avg_price = live_prices.mean()
    gap_pct = (optimal_price_benchmark / current_avg_price) - 1 if current_avg_price > 0 else 0.0

    if gap_pct > DIRECTION_TOL_PCT:
        direction = 'up'
        candidate_uplifts = UP_RANGE
    elif gap_pct < -DIRECTION_TOL_PCT:
        direction = 'down'
        candidate_uplifts = DOWN_RANGE
    else:
        direction = 'flat'
        candidate_uplifts = FLAT_RANGE

    for uplift_pct in candidate_uplifts:
        treated_prices = live_prices * (1 + uplift_pct)

        # Keep each tested uplift fully inside the modeled price range.
        if treated_prices.max() > MAX_MODELED_PRICE or treated_prices.min() < MIN_MODELED_PRICE:
            continue

        if is_fb:
            x_base = np.column_stack([np.ones(len(live_prices)), live_prices])
            x_treat = np.column_stack([np.ones(len(treated_prices)), treated_prices])
            eta_base = x_base @ new_params
            eta_treat = x_treat @ new_params
            se_eta_treat = np.sqrt(np.einsum('ij,jk,ik->i', x_treat, cov_matrix[:2, :2], x_treat))
        else:
            x_base = np.column_stack([
                np.ones(len(live_prices)),
                live_prices,
                num_conn,
                dur_matrix
            ])
            x_treat = np.column_stack([
                np.ones(len(treated_prices)),
                treated_prices,
                num_conn,
                dur_matrix
            ])
            eta_base = x_base @ new_params
            eta_treat = x_treat @ new_params
            se_eta_treat = np.sqrt(np.einsum('ij,jk,ik->i', x_treat, cov_matrix, x_treat))

        take_rate_base = 1 / (1 + np.exp(-eta_base))
        take_rate_mean = 1 / (1 + np.exp(-eta_treat))
        take_rate_lower = 1 / (1 + np.exp(-(eta_treat - z * se_eta_treat)))
        take_rate_upper = 1 / (1 + np.exp(-(eta_treat + z * se_eta_treat)))

        current_live_revenue = np.mean(live_prices * take_rate_base)
        rev_mean  = np.mean(treated_prices * take_rate_mean)
        rev_lower = np.mean(treated_prices * take_rate_lower)
        rev_upper = np.mean(treated_prices * take_rate_upper)

        opportunity_cost = max(current_live_revenue - rev_mean, 0.001)
        potential_gain = rev_upper - current_live_revenue
        exploration_score = potential_gain / opportunity_cost if potential_gain > 0 else -1

        exploration_results.append({
            'segment': seg,
            'region_group': dur,
            'direction': direction,
            'uplift_pct': uplift_pct,
            'current_avg_price': current_avg_price,
            'treated_avg_price': treated_prices.mean(),
            'optimal_price_benchmark': optimal_price_benchmark,
            'current_live_revenue': current_live_revenue,
            'take_rate_mean': take_rate_mean.mean(),
            'take_rate_lower_5': take_rate_lower.mean(),
            'take_rate_upper_95': take_rate_upper.mean(),
            'expected_revenue': rev_mean,
            'revenue_lower': rev_lower,
            'revenue_upper': rev_upper,
            'max_treated_price': treated_prices.max(),
            'offer_count': len(live_grp),
            'exploration_score': exploration_score
        })
        valid_uplifts += 1

    print(
        f"{seg} × {dur}: direction={direction} | {valid_uplifts} feasible uplifts | "
        f"current avg=${current_avg_price:.2f} | benchmark optimal=${optimal_price_benchmark:.2f} | gap={gap_pct:.1%}"
    )

explore_df = pd.DataFrame(exploration_results)
print(f"\nExploration analysis: {len(explore_df):,} uplift-segment combinations evaluated")
print("Treatment is now defined as a direction-aware percent change on observed live final prices.")
print(f"Modeled final-price range applied: ${MIN_MODELED_PRICE:.2f} to ${MAX_MODELED_PRICE:.2f}")

In [0]:
# Meaningful uplift exploration on the observed live price distribution.
MIN_ABS_UPLIFT_PCT = 0.03  # Require at least a 3% move in either direction
MIN_OFFERS = 5000          # Guardrail: avoid very thin segments dominating the ranking

candidates = explore_df[
    (explore_df['exploration_score'] > 0) &
    (explore_df['uplift_pct'].abs() >= MIN_ABS_UPLIFT_PCT) &
    (explore_df['offer_count'] >= MIN_OFFERS)
].copy()

# Add fitted sample size context from params_df
candidates = candidates.merge(
    params_df[['segment', 'region_group', 'n_obs', 'n_sales']],
    on=['segment', 'region_group'],
    how='left'
)

candidates['revenue_gain_vs_live'] = candidates['expected_revenue'] - candidates['current_live_revenue']
candidates['uplift_pct_label'] = (candidates['uplift_pct'] * 100).round(1)

# Rank by exploration score, then by expected revenue gain vs current live baseline
candidates = candidates.sort_values(
    ['exploration_score', 'revenue_gain_vs_live'],
    ascending=[False, False]
)

# Show top 20 exploration candidates
top_candidates = candidates.head(20)[[
    'segment', 'region_group', 'direction', 'uplift_pct_label', 'current_avg_price', 'treated_avg_price',
    'optimal_price_benchmark', 'take_rate_mean', 'expected_revenue', 'revenue_upper',
    'current_live_revenue', 'revenue_gain_vs_live', 'offer_count', 'n_obs', 'exploration_score'
]].copy()

top_candidates['current_avg_price'] = top_candidates['current_avg_price'].round(2)
top_candidates['treated_avg_price'] = top_candidates['treated_avg_price'].round(2)
top_candidates['optimal_price_benchmark'] = top_candidates['optimal_price_benchmark'].round(2)
top_candidates['take_rate_mean'] = (top_candidates['take_rate_mean'] * 100).round(3)
top_candidates['expected_revenue'] = top_candidates['expected_revenue'].round(4)
top_candidates['revenue_upper'] = top_candidates['revenue_upper'].round(4)
top_candidates['current_live_revenue'] = top_candidates['current_live_revenue'].round(4)
top_candidates['revenue_gain_vs_live'] = top_candidates['revenue_gain_vs_live'].round(4)
top_candidates['exploration_score'] = top_candidates['exploration_score'].round(2)

top_candidates = top_candidates.rename(columns={
    'segment': 'Segment',
    'region_group': 'Region Group',
    'direction': 'Direction',
    'uplift_pct_label': 'Uplift (%)',
    'current_avg_price': 'Current Live Avg ($)',
    'treated_avg_price': 'Treated Avg ($)',
    'optimal_price_benchmark': 'Model Optimal ($)',
    'take_rate_mean': 'Est. Take Rate (%)',
    'expected_revenue': 'E[Rev] at Test ($)',
    'revenue_upper': 'Rev Upper Bound ($)',
    'current_live_revenue': 'Current Live Rev ($)',
    'revenue_gain_vs_live': 'Rev Gain vs Live ($)',
    'offer_count': 'Live Offers',
    'n_obs': 'Train Obs',
    'exploration_score': 'Explore Score'
})

print(f"Candidates after filtering (|uplift|>={MIN_ABS_UPLIFT_PCT:.0%}, live offers>={MIN_OFFERS:,}): {len(candidates):,}")
print("\nTop 20 Direction-Aware Percent-Change Candidates on Observed Live Prices:")
display(top_candidates.reset_index(drop=True))

In [0]:
import numpy as np
import pandas as pd

# Recommend direction and exploration step size per segment.
# Goal: turn the model-optimal final price benchmark into practical % arms on the
# current live price distribution, while respecting meaningful $ moves and modeled bounds.

DIRECTION_TOL_PCT = 0.05
MIN_MEANINGFUL_DOLLAR_MOVE = 3.0
TARGET_DOLLAR_MOVE = 5.0
MIN_STEP_PCT = 0.03
MAX_STEP_PCT = 0.15
MAX_ARMS_PER_SIDE = 3
MAX_TEST_PCT = 0.25

MAX_MODELED_PRICE = float(price_grid.max())
MIN_MODELED_PRICE = float(price_grid.min())

recommendation_rows = []

for (seg, rg), opt_row in (
    results_df[results_df['is_optimal']]
    .groupby(['segment', 'region_group'], as_index=False)
):
    live_grp = df_test[
        (df_test['market_traveler_segment'] == seg) &
        (df_test['region_group'] == rg)
    ].copy()
    if len(live_grp) == 0:
        continue

    live_prices = live_grp['price'].astype(float)
    current_avg = float(live_prices.mean())
    current_median = float(live_prices.median())
    current_min = float(live_prices.min())
    current_max = float(live_prices.max())
    optimal_price = float(opt_row['price'].iloc[0])

    gap_pct = (optimal_price / current_avg) - 1 if current_avg > 0 else 0.0
    gap_dollar = optimal_price - current_avg

    if gap_pct > DIRECTION_TOL_PCT:
        direction = 'up'
    elif gap_pct < -DIRECTION_TOL_PCT:
        direction = 'down'
    else:
        direction = 'flat'

    # Convert meaningful dollar moves into % moves using current live average price.
    min_meaningful_step_pct = max(MIN_STEP_PCT, MIN_MEANINGFUL_DOLLAR_MOVE / current_avg)
    target_step_pct = max(min_meaningful_step_pct, TARGET_DOLLAR_MOVE / current_avg)
    recommended_step_pct = min(MAX_STEP_PCT, target_step_pct)

    # Round up to whole percentage points for easier experiment design.
    recommended_step_pct = np.ceil(recommended_step_pct * 100) / 100
    min_meaningful_step_pct = np.ceil(min_meaningful_step_pct * 100) / 100

    max_up_pct = min(MAX_TEST_PCT, (MAX_MODELED_PRICE / current_max) - 1) if current_max > 0 else 0.0
    max_down_pct = max(-MAX_TEST_PCT, (MIN_MODELED_PRICE / current_min) - 1) if current_min > 0 else 0.0

    if direction == 'up':
        candidate_arms = [0.0]
        for k in range(1, MAX_ARMS_PER_SIDE + 1):
            arm = k * recommended_step_pct
            if arm <= max_up_pct + 1e-9:
                candidate_arms.append(round(arm, 2))
    elif direction == 'down':
        candidate_arms = [0.0]
        for k in range(1, MAX_ARMS_PER_SIDE + 1):
            arm = -k * recommended_step_pct
            if arm >= max_down_pct - 1e-9:
                candidate_arms.append(round(arm, 2))
    else:
        candidate_arms = [0.0]
        if recommended_step_pct <= max_up_pct + 1e-9:
            candidate_arms.append(round(recommended_step_pct, 2))
        if -recommended_step_pct >= max_down_pct - 1e-9:
            candidate_arms.append(round(-recommended_step_pct, 2))
        candidate_arms = sorted(set(candidate_arms))

    recommendation_rows.append({
        'segment': seg,
        'region_group': rg,
        'live_offers': len(live_grp),
        'current_live_avg_price': current_avg,
        'current_live_median_price': current_median,
        'current_live_min_price': current_min,
        'current_live_max_price': current_max,
        'model_optimal_price': optimal_price,
        'gap_dollar': gap_dollar,
        'gap_pct': gap_pct,
        'direction': direction,
        'min_meaningful_step_pct': min_meaningful_step_pct,
        'recommended_step_pct': recommended_step_pct,
        'max_feasible_up_pct': max_up_pct,
        'max_feasible_down_pct': max_down_pct,
        'suggested_arms_pct': ', '.join([f'{x*100:+.0f}%' for x in candidate_arms]),
        'suggested_arms_count': len(candidate_arms)
    })

exploration_design_df = pd.DataFrame(recommendation_rows)
exploration_design_df = exploration_design_df.sort_values(['segment', 'region_group']).reset_index(drop=True)

for col in [
    'current_live_avg_price', 'current_live_median_price', 'current_live_min_price',
    'current_live_max_price', 'model_optimal_price', 'gap_dollar'
]:
    exploration_design_df[col] = exploration_design_df[col].round(2)

for col in [
    'gap_pct', 'min_meaningful_step_pct', 'recommended_step_pct',
    'max_feasible_up_pct', 'max_feasible_down_pct'
]:
    exploration_design_df[col] = (exploration_design_df[col] * 100).round(1)

display(
    exploration_design_df.rename(columns={
        'segment': 'Segment',
        'region_group': 'Region Group',
        'live_offers': 'Live Offers',
        'current_live_avg_price': 'Current Avg ($)',
        'current_live_median_price': 'Current Median ($)',
        'current_live_min_price': 'Current Min ($)',
        'current_live_max_price': 'Current Max ($)',
        'model_optimal_price': 'Model Optimal ($)',
        'gap_dollar': 'Gap ($)',
        'gap_pct': 'Gap (%)',
        'direction': 'Direction',
        'min_meaningful_step_pct': 'Min Meaningful Step (%)',
        'recommended_step_pct': 'Recommended Step (%)',
        'max_feasible_up_pct': 'Max Feasible Up (%)',
        'max_feasible_down_pct': 'Max Feasible Down (%)',
        'suggested_arms_pct': 'Suggested Arms',
        'suggested_arms_count': 'Arms Count'
    })
)

print('Interpretation:')
print('- Gap (%) compares model-optimal final price vs current live average final price.')
print('- Direction chooses up/down/flat from that gap.')
print('- Recommended Step (%) converts a meaningful $ move (~$5 target, minimum ~$3) into a % move.')
print('- Suggested Arms are practical experiment arms constrained to the modeled price range.')